In [1]:
# ============================================================
# EX 9: TEXT GENERATION USING LSTM NETWORKS
# AI23531 - Deep Learning
# Dataset: Tiny Shakespeare Corpus
# ============================================================

import tensorflow as tf
import numpy as np

print("TensorFlow version:", tf.__version__)

# ------------------------------------------------------------
# STEP 1: LOAD AND PREPROCESS THE DATASET
# ------------------------------------------------------------

print("\nLoading Shakespeare dataset...")

file_path = tf.keras.utils.get_file(
    "shakespeare.txt",
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
)

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read().lower()

print("Dataset loaded successfully!")
print("Total characters:", len(text))

# Create character vocabulary
chars = sorted(set(text))

c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for i, c in enumerate(chars)}

print("Number of unique characters:", len(chars))


# ------------------------------------------------------------
# STEP 2: CREATE INPUT AND OUTPUT SEQUENCES
# ------------------------------------------------------------

seq_len = 40

X = []
y = []

# Use a smaller portion for faster Colab execution
# Remove this line if you want to use the complete corpus
text_small = text[:200000]

for i in range(len(text_small) - seq_len):
    input_seq = text_small[i:i + seq_len]
    target_char = text_small[i + seq_len]

    X.append([c2i[c] for c in input_seq])
    y.append(c2i[target_char])

X = np.array(X)
y = np.array(y)

print("\nInput shape :", X.shape)
print("Target shape:", y.shape)


# ------------------------------------------------------------
# STEP 3: BUILD THE LSTM MODEL
# ------------------------------------------------------------

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(
        input_dim=len(chars),
        output_dim=64,
        input_length=seq_len
    ),

    tf.keras.layers.LSTM(128),

    tf.keras.layers.Dense(
        len(chars),
        activation="softmax"
    )
])

model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam"
)

print("\nLSTM Model:")
model.summary()


# ------------------------------------------------------------
# STEP 4: TRAIN THE MODEL
# ------------------------------------------------------------

print("\nTraining the LSTM model...")

history = model.fit(
    X,
    y,
    batch_size=128,
    epochs=1,
    verbose=1
)

print("\nTraining completed!")


# ------------------------------------------------------------
# STEP 5: TEXT GENERATION FUNCTION
# ------------------------------------------------------------

def generate(seed, length=300, temperature=0.8):

    seed = seed.lower()

    # Make sure every character exists in vocabulary
    seed = "".join(c for c in seed if c in c2i)

    seq = [c2i[c] for c in seed]

    for _ in range(length):

        # Take the last 40 characters
        inp = np.array(seq[-seq_len:]).reshape(1, -1)

        # Predict next-character probabilities
        pred = model.predict(inp, verbose=0)[0]

        # Temperature controls randomness
        pred = np.log(pred + 1e-8) / temperature
        pred = np.exp(pred)
        pred = pred / np.sum(pred)

        # Select next character
        next_idx = np.random.choice(len(pred), p=pred)

        seq.append(next_idx)

    generated_part = "".join(
        i2c[i] for i in seq[len(seed):]
    )

    return seed + generated_part


# ------------------------------------------------------------
# STEP 6: GENERATE TEXT
# ------------------------------------------------------------

seed_text = "shall i compare thee to a summer's day?\n"

generated_text = generate(
    seed_text,
    length=300,
    temperature=0.8
)

print("\n" + "=" * 60)
print("GENERATED TEXT")
print("=" * 60)

print(generated_text)

print("\n" + "=" * 60)
print("EXPERIMENT RESULT")
print("=" * 60)
print("Dataset          : Tiny Shakespeare")
print("Model            : Character-Level LSTM")
print("Sequence Length  :", seq_len)
print("Embedding Size   : 64")
print("LSTM Units       : 128")
print("Training Epochs  : 1")
print("Generated Length : 300 characters")
print("=" * 60)

TensorFlow version: 2.21.0

Loading Shakespeare dataset...
1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Dataset loaded successfully!
Total characters: 1115394
Number of unique characters: 39

Input shape : (199960, 40)
Target shape: (199960,)

LSTM Model:


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training the LSTM model...
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 193s 122ms/step - loss: 2.2934

Training completed!

GENERATED TEXT
shall i compare thee to a summer's day?

sorcingiu:
onre to thith a shich with sustere the the wand eith at for,
you. s,
the goreed and of me hantit were wich it the mory and vear my to nogbedinglyo' of nuty,
why glope.

coriranus:!

brarest:
i have gis my bras the shoul.

menene themens houst,
il brot not py sice
is you nod. arly to ke p

EXPERIMENT RESULT
Dataset          : Tiny Shakespeare
Model            : Character-Level LSTM
Sequence Length  : 40
Embedding Size   : 64
LSTM Units       : 128
Training Epochs  : 1
Generated Length : 300 characters
